# STATE QC Cross-Attention — Results
Compares QC cross-attention vs baseline for Replogle (SE_R_Rk562) and Tian1921 datasets.

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ── Paths ────────────────────────────────────────────────────────────────────
BASE     = "/dcai/users/hilarn/55_cu_0055/code/enhance_state/results"
RUN_ID   = "31"
CKPT_TAG = "eval_best.ckpt"

# Best run per model type per dataset (selected by pearson_delta)
RUNS = {
    "QC (Replogle)"  : (f"qc_emb_lr1e-4/qc_emb_{RUN_ID}_lr1e-4",               "Replogle"),
    "Base (Replogle)": (f"baseline_lr1e-4/baseline_{RUN_ID}_lr1e-4",             "Replogle"),
    "QC (Tian)"      : (f"qc_emb_Tian_lr1e-5/qc_emb_Tian_{RUN_ID}_lr1e-5",     "Tian"),
    "Base (Tian)"    : (f"baseline_Tian_lr1e-4/baseline_Tian_{RUN_ID}_lr1e-4",   "Tian"),
}

def run_dir(run_path):
    return os.path.join(BASE, RUN_ID, run_path)

def eval_dir(run_path):
    return os.path.join(run_dir(run_path), CKPT_TAG)

print("Checking directories...")
for label, (path, _) in RUNS.items():
    exists = os.path.isdir(eval_dir(path))
    print(f"  {'OK' if exists else 'MISSING':7s}  {label}  →  {eval_dir(path)}")

## 1 · Training curves

In [ ]:
def load_training_log(run_path):
    path = os.path.join(run_dir(run_path), "version_0", "metrics.csv")
    if not os.path.exists(path):
        return None
    return pd.read_csv(path)

COLORS = {"QC": "#1f77b4", "Base": "#ff7f0e"}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (metric, metric_label) in zip(axes, [("train_loss", "Training loss"), ("val_loss", "Validation loss")]):
    ax.set_title(metric_label)
    ax.set_xlabel("Step")
    ax.set_ylabel(metric_label)
    for label, (path, dataset) in RUNS.items():
        df = load_training_log(path)
        if df is None or metric not in df.columns:
            continue
        sub = df[["step", metric]].dropna()
        model = "QC" if "QC" in label else "Base"
        ls = "-" if "Replogle" in label else "--"
        ax.plot(sub["step"], sub[metric], label=label, color=COLORS[model], ls=ls, lw=1.8)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(BASE, RUN_ID, "training_curves.pdf"), dpi=150)
plt.show()

## 2 · Aggregate evaluation metrics

In [ ]:
def load_agg_metrics(run_path):
    """Load mean row from all *_agg_results.csv files in the eval dir."""
    ed = eval_dir(run_path)
    files = glob.glob(os.path.join(ed, "*_agg_results.csv"))
    if not files:
        return None
    dfs = []
    for f in files:
        df = pd.read_csv(f)
        # The CSV has a 'statistic' column — keep only the mean row
        if "statistic" in df.columns:
            df = df[df["statistic"] == "mean"].drop(columns="statistic")
        df["cell_type"] = os.path.basename(f).replace("_agg_results.csv", "")
        dfs.append(df)
    return pd.concat(dfs, ignore_index=True)

agg_all = []
for label, (name, dataset) in RUNS.items():
    df = load_agg_metrics(name)
    if df is not None:
        df["run"]     = label
        df["dataset"] = dataset
        df["model"]   = "QC" if "QC" in label else "Baseline"
        agg_all.append(df)
    else:
        print(f"No agg metrics found for: {label}")

if agg_all:
    agg = pd.concat(agg_all, ignore_index=True)
    print(agg[["run", "dataset", "model", "cell_type", "pearson_delta", "de_spearman_sig"]].to_string())
else:
    print("No evaluation results found — run predict first.")

In [ ]:
KEY_METRICS = ["pearson_delta", "de_spearman_sig", "pr_auc", "mse_delta"]
key_metrics = [m for m in KEY_METRICS if m in agg.columns]

fig, axes = plt.subplots(1, len(key_metrics), figsize=(5 * len(key_metrics), 5))
if len(key_metrics) == 1:
    axes = [axes]

for ax, metric in zip(axes, key_metrics):
    plot_df = agg.melt(
        id_vars=["run", "dataset", "model", "cell_type"],
        value_vars=[metric], var_name="metric", value_name="value",
    )
    sns.barplot(
        data=plot_df, x="dataset", y="value", hue="model",
        palette={"QC": "#1f77b4", "Baseline": "#ff7f0e"},
        ax=ax, capsize=0.08,
    )
    ax.set_title(metric, fontsize=11)
    ax.set_xlabel("")
    ax.legend(title="")

plt.suptitle("QC cross-attention vs Baseline", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(BASE, RUN_ID, "eval_metrics.pdf"), dpi=150, bbox_inches="tight")
plt.show()

## 3 · Per-perturbation metrics

In [ ]:
def load_pert_metrics(run_path):
    ed = eval_dir(run_path)
    files = [f for f in glob.glob(os.path.join(ed, "*_results.csv")) if "_agg_" not in f]
    if not files:
        return None
    dfs = []
    for f in files:
        df = pd.read_csv(f)
        df["cell_type"] = os.path.basename(f).replace("_results.csv", "")
        dfs.append(df)
    return pd.concat(dfs, ignore_index=True)

# Peek at columns to find the perturbation column name
_sample_path, _ = list(BEST_RUNS["Replogle"].items())[0]
_sample = load_pert_metrics(_sample_path[1] if isinstance(_sample_path, tuple) else _sample_path)

BEST_RUNS = {
    "Replogle": {
        "QC"      : f"qc_emb_lr1e-4/qc_emb_{RUN_ID}_lr1e-4",
        "Baseline": f"baseline_lr1e-4/baseline_{RUN_ID}_lr1e-4",
    },
    "Tian": {
        "QC"      : f"qc_emb_Tian_lr1e-5/qc_emb_Tian_{RUN_ID}_lr1e-5",
        "Baseline": f"baseline_Tian_lr1e-4/baseline_Tian_{RUN_ID}_lr1e-4",
    },
}

# Find pert column name from first available file
def find_pert_col(df):
    for candidate in ["pert", "perturbation", "gene", "condition", "perturbation_name"]:
        if candidate in df.columns:
            return candidate
    # fall back to first non-numeric, non-cell_type column
    for c in df.columns:
        if c != "cell_type" and not pd.api.types.is_numeric_dtype(df[c]):
            return c
    return None

for dataset, runs in BEST_RUNS.items():
    dfs = {}
    for model_name, run_path in runs.items():
        df = load_pert_metrics(run_path)
        if df is not None:
            dfs[model_name] = df

    if len(dfs) < 2:
        print(f"{dataset}: not enough data to compare")
        continue

    # Detect columns
    pert_col = find_pert_col(dfs["QC"])
    print(f"{dataset}: pert column = '{pert_col}',  columns = {list(dfs['QC'].columns)}")

    num_cols = [c for c in dfs["QC"].columns
                if c not in (pert_col, "cell_type") and pd.api.types.is_numeric_dtype(dfs["QC"][c])]
    metric = "pearson_delta" if "pearson_delta" in num_cols else (num_cols[0] if num_cols else None)
    if metric is None or pert_col is None:
        print(f"{dataset}: cannot find pert column or metric, skipping")
        continue

    merged = dfs["QC"][[pert_col, "cell_type", metric]].merge(
        dfs["Baseline"][[pert_col, "cell_type", metric]],
        on=[pert_col, "cell_type"], suffixes=("_qc", "_base")
    )

    fig, ax = plt.subplots(figsize=(5, 5))
    ax.scatter(merged[f"{metric}_base"], merged[f"{metric}_qc"], alpha=0.4, s=10, color="#1f77b4")
    lims = [min(merged[[f"{metric}_base", f"{metric}_qc"]].min()),
            max(merged[[f"{metric}_base", f"{metric}_qc"]].max())]
    ax.plot(lims, lims, "k--", lw=0.8)
    ax.set_xlabel(f"Baseline — {metric}")
    ax.set_ylabel(f"QC — {metric}")
    ax.set_title(f"{dataset}: per-perturbation {metric}\n(above diagonal = QC better)")
    plt.tight_layout()
    plt.savefig(os.path.join(BASE, RUN_ID, f"scatter_{dataset}.pdf"), dpi=150)
    plt.show()

    n_better = (merged[f"{metric}_qc"] > merged[f"{metric}_base"]).sum()
    print(f"{dataset}: QC better in {n_better}/{len(merged)} perturbations")